# Correlation lag-tau experiments

This notebook calculates the tau-generalized, lagged Pearson-correlation
assignments for the canonical temperature/TSI dyads. It needs the full real and
surrogate time-series inputs from the CCM-software project roots to run.

It's the calculation behind Text S3 and Fig. S8: a deliberately simple linear
alternative to CCM, pushed through as close to the same pipeline as possible (same
dyads, same τ-based resampling, same ±20-timestep lag scan, same phase-randomized
surrogate significance test) so the two methods are genuinely comparable. The
question comes straight from Discussion: *"If TSI were a dominant or direct driver
of climate variability, a simple linear relationship would be expected and
recoverable by simple correlation."* Finding that simple correlation is mostly
insignificant where CCM finds discernible influence (Fig. S10 = CCM skill minus this
correlation) is evidence that the TSI–temperature relationship is real, just better
described in state space than as a fixed linear coupling.


## Workflow

1. Set the experiment scope and enable `RUN_EXPERIMENT`.
2. Compute the optimal correlation lag and surrogate-outperformance fractions.
3. Optionally enable `PERSIST_ASSIGNMENTS` to upsert the compact assignment table
   [Simple Correlation, All Dyads (Figure S8)](2_plot__corr_lag_tau_resultsgrid.ipynb) uses.

The heavy lag-profile and surrogate-profile diagnostic exports are left out on
purpose.

`min_lag=-1` mirrors CCM's own convention: scan both directions, but only report
lags where TSI leads (or is roughly contemporaneous with) temperature, rather than
letting temperature "predict" TSI at a negative lag — see Methods' discussion of
scanning ℓ but reporting the positive-lag optimum.


## Imports and experiment controls


In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

from cedarkit.core.project_config import load_config
from cedarkit.core.data_var import VarObject
from cedarkit.core.data_objects import merge_variable_ts
from cedarkit.utils.io.timeseries_utils import remove_extra_index


In [2]:
# Experiment controls
tau_values = list(range(1, 8))
lag_ts_limit = 20
peak_window_halfwidth = 3
min_lag = -1  # select the maximum |r| among lags >= -1

# Choose a small scope while developing; None means the complete canonical grid.
run_pair_labels = None  # e.g. ['Alley-Wu', 'Tian-Wu']
run_row_categories = ['original', 'lineardetrended', '1kyrfir']
run_tau_values = None  # e.g. [1, 2, 4]

# The experiment is deliberately opt-in: the full run reads every real and
# surrogate series.  Set True after reviewing the controls above.
RUN_EXPERIMENT = False

# Write only the compact assignment table used by the plot notebook.  The
# large real-profile and surrogate-profile diagnostic tables are not produced.
PERSIST_ASSIGNMENTS = False


## Locate source projects


In [3]:
# The experiment needs the full dyad inputs, which live in the CCM-software
# project roots rather than the lightweight Paleobook export.
stem = Path(*(p := Path.cwd().resolve()).parts[: p.parts.index("notebooks")])
_candidates = [stem, Path('/Users/jlanders/PycharmProjects/CCM_software')]
PROJECT_ROOT = next((p for p in _candidates if (p / 'hol_temp_tsi_ccm1k').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not locate the CCM-software project root containing hol_temp_tsi_ccm1k'
    )

PROJECT_ROOTS = [PROJECT_ROOT / 'hol_temp_tsi_ccm1k', PROJECT_ROOT / 'hol_temp_tsi_ccm']
HT_ROOT = PROJECT_ROOT / 'hol_temp_tsi_ccm1k'


def resolve_proj_dir(project_name: str, preferred_roots=None) -> Path:
    roots = preferred_roots if preferred_roots is not None else PROJECT_ROOTS
    proj_dir = next((root / project_name for root in roots if (root / project_name).exists()), None)
    if proj_dir is None:
        root_txt = ', '.join([str(r) for r in roots])
        raise FileNotFoundError(f'Could not locate project {project_name} in [{root_txt}]')
    return proj_dir


## Correlation calculation helpers


In [4]:
def load_surrogate_table(var_obj: VarObject) -> pd.DataFrame:
    surr_path = var_obj.surr_data_dir_path / f'{var_obj.surr_ts_csv}.csv'
    if not surr_path.exists():
        raise FileNotFoundError(f'Missing surrogate table for {var_obj.var_id}: {surr_path}')
    surr_df = pd.read_csv(surr_path)
    surr_df = remove_extra_index(surr_df)
    surr_df, _ = var_obj.standardize_time_var(var_obj.surr_ts_time, surr_df, var_obj.surr_prefix)
    if 'time' not in surr_df.columns:
        raise ValueError(f'Failed to standardize surrogate time column for {var_obj.var_id}: {surr_path}')
    return surr_df


In [5]:
def extract_surr_num(col_name: str):
    m = re.search(r'_(\d+)$', str(col_name))
    return int(m.group(1)) if m else np.nan


def lag_then_sample_corr(pair_df: pd.DataFrame, x_name: str, y_name: str, lag_ts: int, tau: int, phase: int):
    tmp = pair_df[[x_name, y_name]].copy()
    tmp[y_name] = tmp[y_name].shift(lag_ts)
    tmp = tmp.dropna().reset_index(drop=True)
    if len(tmp) < 3:
        return np.nan, len(tmp)
    tmp = tmp.iloc[phase::tau].reset_index(drop=True)
    if len(tmp) < 3:
        return np.nan, len(tmp)
    r = tmp[x_name].corr(tmp[y_name], method='pearson')
    return (np.abs(r) if pd.notna(r) else np.nan), len(tmp)


def compute_corr_profiles(
    pair_real, surr_x, surr_y,
    x_col, y_col, col_var_id, target_var_id,
    tau_values, lag_ts_limit,
    min_lag=-1, peak_window_halfwidth=3,
):
    lag_ts_values = list(range(-lag_ts_limit, lag_ts_limit + 1))

    y_surr_cols = [c for c in surr_y.columns if c != 'time'][1:]
    x_surr_cols = [c for c in surr_x.columns if c != 'time'][1:]

    y_surr_pairs = {
        s_col: pair_real[['time', x_col]].merge(
            surr_y[['time', s_col]].rename(columns={s_col: 'surr_val'}),
            on='time', how='inner'
        )
        for s_col in y_surr_cols
    }
    x_surr_pairs = {
        s_col: pair_real[['time', y_col]].merge(
            surr_x[['time', s_col]].rename(columns={s_col: 'surr_val'}),
            on='time', how='inner'
        )
        for s_col in x_surr_cols
    }

    real_rows, surr_rows, summary_rows = [], [], []

    for tau in tau_values:
        phase_rows = []
        for lag_ts in lag_ts_values:
            for phase in range(tau):
                corr, _ = lag_then_sample_corr(pair_real, x_col, y_col, lag_ts, tau, phase)
                phase_rows.append({'lag_ts': int(lag_ts), 'corr': corr})

        tau_phase = pd.DataFrame(phase_rows).dropna(subset=['corr'])
        if len(tau_phase) == 0:
            continue

        tau_real = tau_phase.groupby('lag_ts', as_index=False)['corr'].mean()
        tau_real['tau'] = tau
        tau_real['phase_agg'] = 'mean_over_phase'
        real_rows.extend(tau_real.to_dict('records'))

        tau_surr_rows = []
        for s_col, pair in y_surr_pairs.items():
            surr_num = extract_surr_num(s_col)
            for lag_ts in lag_ts_values:
                corr, n_obs = lag_then_sample_corr(pair, x_col, 'surr_val', lag_ts, tau, phase=0)
                tau_surr_rows.append({
                    'tau': tau, 'lag_ts': int(lag_ts),
                    'relation': f'{col_var_id}_real__{target_var_id}_surr',
                    'surr_var': target_var_id, 'surr_num': surr_num,
                    'corr': corr, 'n_obs': n_obs,
                })

        for s_col, pair in x_surr_pairs.items():
            surr_num = extract_surr_num(s_col)
            for lag_ts in lag_ts_values:
                corr, n_obs = lag_then_sample_corr(pair, y_col, 'surr_val', lag_ts, tau, phase=0)
                tau_surr_rows.append({
                    'tau': tau, 'lag_ts': int(lag_ts),
                    'relation': f'{col_var_id}_surr__{target_var_id}_real',
                    'surr_var': col_var_id, 'surr_num': surr_num,
                    'corr': corr, 'n_obs': n_obs,
                })

        surr_rows.extend(tau_surr_rows)

        # Select best lag by max |r|, optionally constrained to lag >= min_lag.
        cand = tau_real.dropna(subset=['corr']).copy()
        if min_lag is not None:
            constrained = cand[cand['lag_ts'] >= int(min_lag)]
            if len(constrained) > 0:
                cand = constrained

        if len(cand) == 0:
            continue

        rho = float(cand['corr'].max())
        tied_lags = sorted(cand.loc[cand['corr'] == rho, 'lag_ts'].astype(int).tolist())
        lag = sorted(tied_lags, key=lambda v: (abs(v), v))[0]

        neighbors = tau_real[(tau_real['lag_ts'] >= lag - peak_window_halfwidth) & (tau_real['lag_ts'] <= lag + peak_window_halfwidth) & (tau_real['lag_ts'] != lag)]['corr'].dropna()
        if np.isfinite(rho) and rho != 0 and len(neighbors) > 0:
            peak_score = float(np.clip((rho - float(neighbors.mean())) / rho, 0.0, 1.0))
        else:
            peak_score = np.nan

        tau_surr_df = pd.DataFrame(tau_surr_rows).dropna(subset=['corr'])
        if len(tau_surr_df) > 0:
            surr_best = tau_surr_df.groupby(['surr_var', 'surr_num'], as_index=False)['corr'].max()
        else:
            surr_best = pd.DataFrame(columns=['surr_var', 'surr_num', 'corr'])

        rx = surr_best[surr_best['surr_var'] == col_var_id]
        ry = surr_best[surr_best['surr_var'] == target_var_id]
        rx_frac = float((rx['corr'] > rho).sum() / len(rx)) if len(rx) > 0 else None
        ry_frac = float((ry['corr'] > rho).sum() / len(ry)) if len(ry) > 0 else None

        summary_rows.append({
            'tau': tau,
            'lag': int(lag),
            'rho': rho,
            'tied_lags': tied_lags,
            'has_tie': len(tied_lags) > 1,
            'peak_score': peak_score,
            'surr_rx_outperforming_frac': rx_frac,
            'surr_ry_outperforming_frac': ry_frac,
        })

    return (
        pd.DataFrame(real_rows),
        pd.DataFrame(surr_rows),
        pd.DataFrame(summary_rows).sort_values('tau').reset_index(drop=True),
    )


## Canonical dyads and batch helpers


In [6]:
pair_specs = [
    ('Tian-Wu', 'Tian22HT115kaALLGMST_Wu18TSI', 'Tian22HT115kaALLGMSTLinear_Wu18TSILinear', 'tian22ht115kaallgmst1kyrfir_wu18tsianom1kyrfir', False),
    ('Erb-Wu', 'Erb22daGMST_Wu18TSI', 'Erb22daGMSTLinear_Wu18TSILinear', 'erb22dagmst1kyrfir_wu18tsianom1kyrfir', False),
    ('Erb-Vieira', 'Erb22daGMST_Vieira11TSI', 'Erb22daGMSTLinear_Vieira11TSILinear', 'erb22dagmst1kyrfir_vieira11tsianom1kyrfir', False),
    ('Alley-Wu', 'GISP2Alley00Tanom_Wu18TSI', 'GISP2Alley00TanomLinear_Wu18TSILinear', 'alley00gisp2multiproxy1kyrfir_wu18tsianom1kyrfir', False),
    ('Alley-Vieira', 'GISP2Alley00Tanom_Vieira11TSI', 'GISP2Alley00TanomLinear_Vieira11TSILinear', 'alley00gisp2multiproxy1kyrfir_vieira11tsianom1kyrfir', False),
    ('Doering-Wu', 'GISP2Doering22T15N_Wu18TSI', 'GISP2Doering22T15NLinear_Wu18TSILinear', 'doering22gisp2t15n1kyrfir_wu18tsianom1kyrfir', True),
    ('GISP2 Seierstad-Wu', 'GISP2Seierstad14d18O_Wu18TSI', 'GISP2Seierstad14d18OLinear_Wu18TSILinear', 'seierstad14gisp2d18o1kyrfir_wu18tsianom1kyrfir', False),
    ('NGRIP Seierstad-Wu', 'NGRIP1Seierstad14d18O_Wu18TSI', 'NGRIP1Seierstad14d18OLinear_Wu18TSILinear', 'seierstad14ngrip1d18o1kyrfir_wu18tsianom1kyrfir', False),
    ('GISP2 Martin-Wu', 'GISP2Martin24Tanom_Wu18TSI', 'GISP2Martin24TanomLinear_Wu18TSILinear', 'martin24gisp2tanom1kyrfir_wu18tsianom1kyrfir', False),
    ('NGRIP Martin-Wu', 'NGRIPMartin24Tanom_Wu18TSI', 'NGRIPMartin24TanomLinear_Wu18TSILinear', 'martin24ngriptanom1kyrfir_wu18tsianom1kyrfir', False),
]


In [7]:
row_triplets = [
    ('original', 'hol_temp_tsi_ccm', 1),
    ('lineardetrended', 'hol_temp_tsi_ccm', 2),
    ('1kyrfir', 'hol_temp_tsi_ccm1k', 3),
]

if 'corr_tau_cache' not in globals():
    corr_tau_cache = {}


def build_run_specs(run_pair_labels=None, run_row_categories=None):
    if run_row_categories is None:
        run_row_categories = ['original', 'lineardetrended', '1kyrfir']

    allowed_pairs = set(run_pair_labels) if run_pair_labels is not None else None
    allowed_rows = set(run_row_categories)

    run_rows = []
    skipped_rows = []

    for pair_label, orig_name, linear_name, one_k_name, _require_orig in pair_specs:
        if (allowed_pairs is not None) and (pair_label not in allowed_pairs):
            continue

        names = [orig_name, linear_name, one_k_name]

        for row_cat, root_name, idx in row_triplets:
            if row_cat not in allowed_rows:
                continue

            proj_name = names[idx - 1]
            proj_dir = PROJECT_ROOT / root_name / proj_name
            if not proj_dir.exists():
                skipped_rows.append({
                    'pair_label': pair_label,
                    'row_category': row_cat,
                    'proj_root': root_name,
                    'proj_name': proj_name,
                    'reason': 'project_missing',
                })
                continue

            run_rows.append({
                'pair_label': pair_label,
                'row_category': row_cat,
                'proj_root': root_name,
                'proj_name': proj_name,
            })

    run_specs_df = pd.DataFrame(run_rows)
    skipped_specs_df = pd.DataFrame(skipped_rows)
    return run_specs_df, skipped_specs_df


def compute_corr_tau_tables_cached(proj_root, proj_name, tau_values, lag_ts_limit, min_lag=-1):
    key = (
        proj_root,
        proj_name,
        tuple(sorted([int(t) for t in tau_values])),
        int(lag_ts_limit),
        int(min_lag) if min_lag is not None else None,
    )
    if key in corr_tau_cache:
        return corr_tau_cache[key], True

    print(f'Computing corr/lag tables for {proj_name} (tau={tau_values}, lag_ts_limit={lag_ts_limit}, min_lag={min_lag})...')
    out = compute_corr_tau_tables(
        proj_root,
        proj_name,
        tau_values=tau_values,
        lag_ts_limit=lag_ts_limit,
        min_lag=min_lag,
    )
    corr_tau_cache[key] = out
    return out, False


In [8]:
def compute_corr_tau_tables(proj_root, proj_name, tau_values, lag_ts_limit, min_lag=-1):
    """
    Return lag/rho tables for one project with lag-first semantics.
    Outputs:
      - lag_real_df: (tau, lag, rho) phase-averaged real profile
      - lag_surr_df: (tau, lag, surr_var, surr_num, rho) surrogate profiles
      - summary_df: per-tau optimal lag/rho and surrogate outperform fractions
    """
    try:
        proj_dir = resolve_proj_dir(proj_name, preferred_roots=[PROJECT_ROOT / proj_root])
    except FileNotFoundError:
        return None, None, None

    proj_cfg_path = proj_dir / 'proj_config.yaml'
    if not proj_cfg_path.exists():
        return None, None, None

    proj_config = load_config(proj_cfg_path)
    col_var_id = proj_config.get_dynamic_attr('{var}.var_id', 'col')
    target_var_id = proj_config.get_dynamic_attr('{var}.var_id', 'target')

    var_obj_x = VarObject(proj_config, proj_dir=proj_dir, var_id=col_var_id)
    var_obj_y = VarObject(proj_config, proj_dir=proj_dir, var_id=target_var_id)

    var_obj_x.get_real()
    var_obj_y.get_real()
    if var_obj_x.ts is None or var_obj_y.ts is None:
        return None, None, None

    x_col_local = var_obj_x.col_name
    y_col_local = var_obj_y.col_name
    pair_real = merge_variable_ts(var_obj_x, var_obj_y).rename(columns={var_obj_x.var: x_col_local, var_obj_y.var: y_col_local})
    if len(pair_real) < 3:
        return None, None, None

    surr_x = load_surrogate_table(var_obj_x)
    surr_y = load_surrogate_table(var_obj_y)

    var_x_label = getattr(var_obj_x, 'var', col_var_id)
    var_y_label = getattr(var_obj_y, 'var', target_var_id)

    real_agg_inner, surr_lag_inner, summary_inner = compute_corr_profiles(
        pair_real=pair_real,
        surr_x=surr_x,
        surr_y=surr_y,
        x_col=x_col_local,
        y_col=y_col_local,
        col_var_id=col_var_id,
        target_var_id=target_var_id,
        tau_values=tau_values,
        lag_ts_limit=lag_ts_limit,
        min_lag=min_lag,
        peak_window_halfwidth=peak_window_halfwidth,
    )

    if len(summary_inner) == 0:
        return None, None, None

    lag_real_df_local = real_agg_inner.rename(columns={'lag_ts': 'lag', 'corr': 'rho'})
    lag_surr_df_local = surr_lag_inner.rename(columns={'lag_ts': 'lag', 'corr': 'rho'})
    summary_df_local = summary_inner.copy()
    summary_df_local['var_x'] = var_x_label
    summary_df_local['var_y'] = var_y_label

    lag_values = set(range(-lag_ts_limit, lag_ts_limit + 1))
    for tau in tau_values:
        sub = lag_real_df_local[lag_real_df_local['tau'] == tau]
        if len(sub) == 0:
            continue
        assert set(sub['lag'].astype(int).tolist()) == lag_values, f'dense lag grid missing for tau={tau} in {proj_name}'

    return lag_real_df_local, lag_surr_df_local, summary_df_local


## Run the selected experiment


In [9]:
run_assignment_corr_df = None

if not RUN_EXPERIMENT:
    print('Experiment not run (RUN_EXPERIMENT=False).')
else:
    run_tau_values = tau_values if run_tau_values is None else sorted(int(t) for t in run_tau_values)
    run_specs_df, skipped_specs_df = build_run_specs(
        run_pair_labels=run_pair_labels,
        run_row_categories=run_row_categories,
    )
    print('Planned run specs:', len(run_specs_df))
    if len(skipped_specs_df) > 0:
        print('Skipped specs:', len(skipped_specs_df))
        display(skipped_specs_df.head(20))
    if len(run_specs_df) == 0:
        raise RuntimeError('No dyad specs selected. Adjust run_pair_labels/run_row_categories.')

    summary_chunks = []
    processed = []
    cache_hits = 0
    for _, spec in run_specs_df.iterrows():
        pair_label, row_cat = spec['pair_label'], spec['row_category']
        root_name, proj_name = spec['proj_root'], spec['proj_name']
        (_, _, summary_df), was_cached = compute_corr_tau_tables_cached(
            root_name, proj_name,
            tau_values=run_tau_values,
            lag_ts_limit=lag_ts_limit,
            min_lag=min_lag,
        )
        if summary_df is None:
            continue
        cache_hits += int(was_cached)
        summary_df = summary_df.copy()
        for key, value in {
            'pair_label': pair_label,
            'row_category': row_cat,
            'proj_root': root_name,
            'proj_name': proj_name,
            'relationship_id': 'r1',
            'E': 1,
        }.items():
            summary_df[key] = value
        summary_chunks.append(summary_df)
        processed.append((pair_label, row_cat, root_name, proj_name))

    if not summary_chunks:
        raise RuntimeError('No projects produced correlation summaries.')

    run_assignment_corr_df = pd.concat(summary_chunks, ignore_index=True)[[
        'pair_label', 'row_category', 'proj_root', 'proj_name', 'relationship_id',
        'var_x', 'var_y', 'E', 'tau', 'lag', 'rho',
        'surr_rx_outperforming_frac', 'surr_ry_outperforming_frac',
        'peak_score', 'has_tie', 'tied_lags',
    ]].copy()
    run_assignment_corr_df['role'] = 'main'
    run_assignment_corr_df['criterium_label'] = 'unrestricted'
    run_assignment_corr_df['is_plausible'] = True
    run_assignment_corr_df = run_assignment_corr_df.sort_values(
        ['pair_label', 'row_category', 'E', 'tau']
    ).reset_index(drop=True)

    required = {
        'pair_label', 'row_category', 'proj_root', 'proj_name', 'relationship_id',
        'var_x', 'var_y', 'E', 'tau', 'lag', 'rho',
        'surr_rx_outperforming_frac', 'surr_ry_outperforming_frac',
        'peak_score', 'has_tie', 'tied_lags', 'role', 'criterium_label', 'is_plausible',
    }
    assert required.issubset(run_assignment_corr_df.columns)
    assert not any(c.endswith('_pos') for c in run_assignment_corr_df.columns)
    print('Processed project rows:', len(processed), '| cache hits:', cache_hits)
    print('Assignment rows:', len(run_assignment_corr_df))


Experiment not run (RUN_EXPERIMENT=False).


## Optionally update the compact plot input


In [10]:
assignment_path = HT_ROOT / 'mixed' / 'corr_summarygrid_inputs_absr' / 'corr_assignment_summarygrid.csv'

if not PERSIST_ASSIGNMENTS:
    print(f'Assignment table not written (PERSIST_ASSIGNMENTS=False): {assignment_path}')
elif run_assignment_corr_df is None or run_assignment_corr_df.empty:
    raise RuntimeError('Nothing to persist. Set RUN_EXPERIMENT=True and run the experiment first.')
else:
    assignment_path.parent.mkdir(parents=True, exist_ok=True)
    key_cols = [
        'pair_label', 'row_category', 'proj_root', 'proj_name', 'relationship_id',
        'E', 'tau', 'role', 'criterium_label',
    ]
    incoming = run_assignment_corr_df.copy()
    if assignment_path.exists():
        existing = pd.read_csv(assignment_path)
        existing_key = existing[key_cols].fillna('').astype(str).agg('|'.join, axis=1)
        incoming_key = incoming[key_cols].fillna('').astype(str).agg('|'.join, axis=1)
        existing = existing.loc[~existing_key.isin(set(incoming_key))]
        combined = pd.concat([existing, incoming], ignore_index=True)
    else:
        combined = incoming
    combined = combined.sort_values(['pair_label', 'row_category', 'E', 'tau']).reset_index(drop=True)
    combined.to_csv(assignment_path, index=False)
    print(f'Wrote {len(combined)} rows: {assignment_path}')


Assignment table not written (PERSIST_ASSIGNMENTS=False): /Users/jlanders/PycharmProjects/CCM_software/hol_temp_tsi_ccm1k/mixed/corr_summarygrid_inputs_absr/corr_assignment_summarygrid.csv
